# Lekse 10 - AI-agenter i produksjon

I denne leksen vil du lære **produksjonsmønstre** for AI-agenter ved bruk av Microsoft Agent Framework (Python). Vi dekker:

- **Observerbarhet** — legge til tidsmåling og logging i agentinteraksjoner
- **Evaluering** — bruke en evalueringsagent for å score responskvalitet
- **Kostnadsstyring** — strategier for tokenoptimalisering og modellvalg

Scenarioet er en **reiseagent** som hjelper brukere med å planlegge turer, med overvåking og evaluering lagt på toppen.


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Produksjonshensyn

Å flytte AI-agenter fra prototyper til produksjon krever nøye oppmerksomhet til tre søyler:

1. **Observerbarhet** — Du trenger innsikt i hva agenten gjør, hvor lang tid det tar, og hvilke verktøy den bruker. Uten sporing og logging er det nesten umulig å feilsøke produksjonsproblemer.

2. **Evaluering** — Automatiserte kvalitetskontroller sikrer at agentens svar forblir nøyaktige, fullstendige og hjelpsomme over tid. En evalueringsagent kan vurdere svarene opp mot definerte kriterier.

3. **Kostnadsstyring** — Token-bruk påvirker direkte kostnadene. Strategier som optimalisering av prompt, valg av modell og caching hjelper med å holde utgiftene under kontroll uten å ofre kvalitet.


## Bygge en observerbar agent

Vi definerer reiseverktøy og pakker agentkallet med timing slik at vi kan overvåke latenstid. I produksjon vil du integrere med OpenTelemetry eller en lignende sporingsbackend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kostnadsstyringsstrategier

Å kontrollere kostnader er avgjørende for produksjons-AI-agenter. Her er nøkkelstrategier:

| Strategi | Beskrivelse |
|---|---|
| **Optimalisering av prompt** | Hold systeminstruksjoner korte. Fjern overflødig kontekst for å redusere input-tokener. |
    "| **Modellvalg** | Bruk mindre, rimeligere modeller (f.eks. GPT-5-mini) for enkle oppgaver som klassifisering eller ekstraksjon, og reserver større modeller til kompleks resonnement. |\n",
| **Caching** | Cache verktøyresultater og hyppige forespørsler for å unngå overflødige API-kall. |
| **Token-budsjetter** | Sett `max_tokens` grenser for å forhindre uventet lange svar. |
| **Batching** | Gruppér flere brukerforespørsler i ett enkelt API-kall der det er mulig. |

I praksis fungerer en lagdelt tilnærming bra: rut enkelt forespørsler til en rask, rimelig modell og eskaler kun komplekse forespørsler til en mer kapabel modell.


## Sammendrag

I denne leksjonen lærte du hvordan du:

1. **Legger til observerbarhet** i agentinteraksjoner med timing og logging, som legger grunnlaget for sporing og overvåking.
2. **Evaluerer agentresponsene** automatisk ved hjelp av en evalueringsagent som vurderer fullstendighet, nøyaktighet og hjelpsomhet.
3. **Håndterer kostnader** gjennom promptoptimalisering, modellvalg, caching og budsjett for tokens.

Disse produksjonsmønstrene bidrar til å sikre at AI-agentene dine er pålitelige, målbare og kostnadseffektive i stor skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
